# 03 — Gate 0 validation: does the stopping rule do what the protocol says?

Run **after** the arm solves from `02_solve.ipynb` (a0–a3, optional a4). Reads each arm's
`run_summary.json` + `portfolio_representation.csv` and renders the verdict tables for the
report-back. No solve happens here.

**G0 pass conditions** (spec Gate 0):
1. targeted carbon capture lands **at** its target (not above) — the target is a stopping rule,
   and `w = t` means it must not act as anything else;
2. the freed budget **visibly reallocates** — candidate destinations: climate corridors (0.96×
   area share in the control era) and the nine EFGs below area share;
3. *(a4, optional)* the pull-invariance arm reproduces the control **exactly** — the empirical
   proof that only `w/t` and the stopping point matter.

Fail branch (spec): revisit θ or the R2 test before any ensemble work — a conversation for the
chat, not a notebook edit.

**Kernel:** `Python (y2y-geo)`. Ethan runs; Claude never executes cells.

In [ ]:
# ---- Setup + locate the arm runs ---------------------------------------------
import sys, pathlib
_cands = [p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
          if (p / "config.py").exists()]
assert _cands, f"config.py not found above {pathlib.Path.cwd()} -- run this notebook from inside the repo"
ROOT = _cands[0]
sys.path.insert(0, str(ROOT))

import importlib, json
import numpy as np
import pandas as pd

import config, leverage_core as lc, ensemble_core as ec
importlib.reload(config); importlib.reload(lc); importlib.reload(ec)

ARMS = ["a0_control", "a1_protocol", "a2_flat30", "a3_flat40", "a4_pullcheck"]  # a4 optional
RUNS = {}
for a in ARMS:
    d = config.RESULTS_DIR / f"iter7_y2y_{a}"
    if (d / "run_summary.json").exists():
        RUNS[a] = d
    else:
        print(f"  (no solved run for {a} -- {'OPTIONAL, skipped' if a == 'a4_pullcheck' else 'REQUIRED, solve it in 02_solve'})")
assert all(a in RUNS for a in ARMS[:4]), "solve the four required arms in 02_solve before running this notebook"
print("arms found:", ", ".join(RUNS))

def rep(d):
    t = pd.read_csv(d / "portfolio_representation.csv").set_index("feature")["relative_held"]
    return t
def summ(d):
    return json.loads((d / "run_summary.json").read_text())
CAPTURE = {a: rep(d) for a, d in RUNS.items()}
SUMMARY = {a: summ(d) for a, d in RUNS.items()}

## Verdict 1 — do the targets bind, and what does "bind" mean?

`w = t` on every arm, so a targeted pool's shortfall must reach zero. **Measured nuance (from the
superseded r1 run): min-shortfall never penalizes *exceeding* a target** — a satiated feature can
free-ride on cells selected for other values (r1: biomass landed at 0.259 against a 0.066 target,
purely incidentally, while m_soc parked at exactly 0.3320, its kink). So the verdicts are:

- **AT target (kink)** — capture ≈ target: the feature's residual cells weren't worth taking for
  anything else;
- **ABOVE target — incidental co-capture (expected)** — the excess is *free-riding*, reported as a
  number, not a failure;
- **BELOW target — investigate** — the only true failure: the stopping point wasn't reached.


In [ ]:
# ---- capture vs target, per arm ------------------------------------------------
TOL = 0.005                     # LP tolerance on captured fraction
POOLS = ["irrecoverable_carbon_m_soc", "irrecoverable_carbon_biomass"]
rows = []
for a in RUNS:
    p = SUMMARY[a]["params"]
    tgts = p.get("targets") or {}
    for f in POOLS:
        t = float(tgts.get(f, 1.0))
        c = float(CAPTURE[a].get(f, np.nan))
        verdict = ("free (no target)"                 if t >= 0.999 else
                   "AT target (kink)"                 if abs(c - t) <= TOL else
                   f"ABOVE target -- incidental co-capture (+{c-t:.3f}, expected)" if c > t else
                   "BELOW target  <-- INVESTIGATE (stopping point not reached)")
        rows.append(dict(arm=a, feature=f.replace("irrecoverable_carbon_", ""),
                         target=round(t, 3), captured=round(c, 3),
                         delta=round(c - t, 4), verdict=verdict,
                         solve_s=round(SUMMARY[a]["solve_seconds"], 1)))
v1 = pd.DataFrame(rows)
print(v1.to_string(index=False))
bad = v1[v1.verdict.str.contains("INVESTIGATE")]
print("\nG0 condition 1:", "PASS -- every targeted pool reached its stopping point (excess above"
      " target, where present, is incidental co-capture and is REPORTED, not penalized)"
      if bad.empty else f"FAIL -- {len(bad)} pool(s) BELOW target; see above")

## Verdict 2 — where did the freed budget go?

Full per-feature capture across arms, as deltas from the control. The prediction: the ~12–16
percentage points carbon gives up reappear in the under-served features — climate corridors and
the nine sub-area-share EFGs are the named candidates.

In [ ]:
# ---- per-feature capture deltas vs control -------------------------------------
cont = lc.continuous_features()
efg_all = [f for f in CAPTURE["a0_control"].index if f not in cont
           and f != "irrecoverable_carbon_sl_soc"]
# the nine EFGs below area share in the CONTROL are the named candidates
efg_low = [f for f in efg_all if CAPTURE["a0_control"][f] < config.BUDGET_PCT]

tbl = pd.DataFrame({a: CAPTURE[a] for a in RUNS if a != "a4_pullcheck"})
out = tbl.loc[cont].copy()
out.loc["EFG mean (all)"] = tbl.loc[efg_all].mean()
out.loc[f"EFG mean (the {len(efg_low)} below area share)"] = tbl.loc[efg_low].mean()
for a in out.columns:
    if a != "a0_control":
        out[f"d {a}"] = out[a] - out["a0_control"]
print(out.round(3).to_string())
print(f"\nreading guide: positive deltas outside carbon = the reallocation the protocol predicts;")
print(f"climate_corridors sat at 0.96x area share and the {len(efg_low)} low EFGs at a control mean of "
      f"{tbl.loc[efg_low, 'a0_control'].mean():.3f}.")

## Verdict 3 — how much does the map move, and (optional) the pull-invariance proof

In [ ]:
# ---- selected-set Jaccard vs control + the a4 exact-reproduction check ---------
def sel(d):
    a = ec._alloc(d / "portfolio.tif")
    return (np.nan_to_num(a) > 0.5)
S0 = sel(RUNS["a0_control"])
print(f"{'arm':<14}{'Jaccard vs a0':>15}{'selected cells':>16}")
for a, d in RUNS.items():
    if a == "a0_control":
        continue
    S = sel(d)
    j = (S & S0).sum() / max((S | S0).sum(), 1)
    print(f"{a:<14}{j:15.4f}{int(S.sum()):16,}")

if "a4_pullcheck" in RUNS:
    S4 = sel(RUNS["a4_pullcheck"])
    same = bool((S4 == S0).all())
    print(f"\nG-uniform (a4): reproduces a0 exactly -> {same}")
    print("   " + ("PASS -- w/t is the only pull parameter; the target below cap_max is inert, as derived"
                   if same else
                   "FAIL -- the w/t equivalence claim is WRONG somewhere; stop and report before Gate 1"))
else:
    print("\n(a4_pullcheck not solved -- optional; solve it in 02_solve to close gate G-uniform empirically)")

## Report-back package

Everything the chat review needs: the frozen T2 (from notebook 01), F8, and the three verdict
tables above, plus `solve_seconds` per arm (the number that prices the future ensemble). **Stop
here** — Gate 1 (S0 construction), the manifest freeze, and all Gurobi work wait on that review.